In [3]:
import transformers

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
phrase = "Thélyson Orélien a eu recours à l'IA ?"
print(tokenizer.tokenize(phrase))
print(tokenizer(phrase))

['T', '##h', '##é', '##ly', '##son', 'Or', '##é', '##lie', '##n', 'a', 'e', '##u', 're', '##co', '##urs', 'à', 'l', "'", 'I', '##A', '?']
{'input_ids': [101, 157, 1324, 2744, 1193, 2142, 2926, 2744, 7174, 1179, 170, 174, 1358, 1231, 2528, 7719, 246, 181, 112, 146, 1592, 136, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [12]:
tokens = tokenizer.tokenize(phrase)
print(tokens)

['T', '##h', '##é', '##ly', '##son', 'Or', '##é', '##lie', '##n', 'a', 'e', '##u', 're', '##co', '##urs', 'à', 'l', "'", 'I', '##A', '?']


In [ ]:
# Basé sur le nombre de bit : vocabulaire de 256 octets
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
print(tokenizer.tokenize(phrase))

['Th', 'Ã©', 'ly', 'son', 'ĠOr', 'Ã©', 'l', 'ien', 'Ġa', 'Ġe', 'u', 'Ġrec', 'ours', 'ĠÃł', 'Ġl', "'", 'IA', 'Ġ?']


In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    ["Elle est vraiment super sa présentation.",
     "Elle est vraiment nulle sa présentation.",
     "Elle est vraiment pourrie sa présentation."])

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1415.12it/s]


[{'label': 'POSITIVE', 'score': 0.9905130863189697}]

In [ ]:
classifier("Elle est vraiment super sa présentation.")

[{'label': 'POSITIVE', 'score': 0.9862370491027832}]

# Cours du MVA de Nathanaël Fijalkow.

In [ ]:
import tiktoken
import numpy as np

enc = tiktoken.get_encoding("gpt2")

words = [
    "king", "queen", "man", "woman", "dog", "cat",
]

embeddings = np.zeros((6, 768))
for w_idx, w in enumerate(words):
    ids = enc.encode(w)
    print(f"{w!r:12} -> token ids {ids}")
    embeddings[w_idx] = E[enc.encode(w)].mean(axis=0)

'king'       -> token ids [3364]
'queen'      -> token ids [4188, 268]
'man'        -> token ids [805]
'woman'      -> token ids [8580]
'dog'        -> token ids [9703]
'cat'        -> token ids [9246]


In [ ]:
import torch
from transformers import GPT2Model

model = GPT2Model.from_pretrained("gpt2")
model.eval()

E = model.wte.weight.detach().numpy()
print(f"Embedding matrix: {E.shape}")
print(
    f"Example embedding for 'king': {E[enc.encode('king')].mean(axis=0)[:5]} ...")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2024.36it/s]

Embedding matrix: (50257, 768)


NameError: name 'enc' is not defined

In [ ]:
class MLPLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids):
        # token_ids : (batch, seq_len)
        x = self.embed(token_ids)      # (batch, seq_len, embed_dim)
        x = self.fc1(x)                # (batch, seq_len, hidden_dim)
        x = F.relu(x)                  # (batch, seq_len, hidden_dim)
        x = self.fc2(x)                # (batch, seq_len, vocab_size)
        return x

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

small_embed = nn.Embedding(5, 3)
linear = nn.Linear(5, 3, bias=False)
linear.weight = nn.Parameter(small_embed.weight.T)  # Transpose to match

idx = torch.tensor([2])
one_hot = F.one_hot(idx, num_classes=5).float()

print(f"Embedding lookup: {small_embed(idx)}")
print(f"Linear on one-hot: {linear(one_hot)}")

In [4]:
class SelfAttentionHead(nn.Module):
    def __init__(self, d_in, d_head):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_head, bias=False)
        self.W_k = nn.Linear(d_in, d_head, bias=False)
        self.W_v = nn.Linear(d_in, d_head, bias=False)
        self.scale = d_head ** 0.5

    def forward(self, x, mask=None):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        scores = (Q @ K.transpose(-2, -1)) / self.scale
        weights = F.softmax(scores, dim=-1)
        return weights @ V

In [5]:
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("Causal mask:")
print(causal_mask)

# Self-attention with causal mask
x = torch.randn(1, seq_len, 8)
head = SelfAttentionHead(8, 4)
ctx, attn_weights = head(x, mask=causal_mask)
print("Attention weights with causal mask:")
print(attn_weights[0].detach().numpy().round(3))

Causal mask:
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])


NameError: name 'F' is not defined

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        head_dim = embed_dim // num_heads
        # One independent SelfAttentionHead per head
        self.heads = nn.ModuleList([
            SelfAttentionHead(embed_dim, head_dim) for _ in range(num_heads)
        ])
        self.W_out = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x, mask=None):
        # Run all heads in parallel, concatenate their outputs
        head_outputs = [head(x, mask)[0] for head in self.heads]
        concat = torch.cat(head_outputs, dim=-1)  # (batch, seq_len, embed_dim)
        return self.W_out(concat)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim), nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask=mask)   # communication
        x = x + self.mlp(self.ln2(x))               # computation
        return x

NameError: name 'TransformerBlock' is not defined

In [ ]:
import torch
from torch import nn


class DecoderTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_layers, n_heads):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(4096, embed_dim)
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, n_heads) for _ in range(n_layers)])
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, input_ids):
        T = input_ids.shape[1]
        x = self.token_emb(input_ids) + self.pos_emb(torch.arange(T))
        mask = torch.tril(torch.ones(T, T))
        for block in self.blocks:
            x = block(x, mask=mask)
        return self.head(x)